In [0]:
from pyspark.sql.types import (
    StringType, DoubleType, LongType, DateType,
    IntegerType, FloatType, ArrayType, DecimalType
)

In [0]:
mdb= spark.table("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.final_mdb_oe_2_0_all_instances_v1")

In [0]:
mdb.limit(2).display()

In [0]:
mdb.printSchema()

In [0]:
from collections import defaultdict
from pyspark.sql import Row

# Create dictionary of datatype -> columns
dtype_dict = defaultdict(list)

for col, dtype in mdb.dtypes:
    dtype_dict[dtype].append(col)

# Convert to list of Rows
data = [
    Row(datatype=dtype, columns=cols)
    for dtype, cols in dtype_dict.items()
]

# Create PySpark DataFrame
dtype_df = spark.createDataFrame(data)

# Display result
dtype_df.display()

In [0]:
from pyspark.sql import functions as F

# Total row count
total_rows = mdb.count()

# Find columns with all NULL values
null_check = mdb.select([
    F.sum(F.col(c).isNotNull().cast("int")).alias(c)
    for c in mdb.columns
])

# Get result as dictionary
non_null_counts = null_check.collect()[0].asDict()

# Columns having fill rate = 0 (all nulls)
all_null_cols = [
    col for col, count in non_null_counts.items()
    if count == 0
]

print("Columns with all NULL values:")
print(all_null_cols)

In [0]:
fill_rate_df = spark.createDataFrame([
    (col, count / total_rows if total_rows > 0 else 0)
    for col, count in non_null_counts.items()
], ["column_name", "fill_rate"])

fill_rate_df.display()

In [0]:
from pyspark.sql import functions as F

# Total row count
total_rows = mdb.count()

# Count non-null values for each column
null_check = mdb.select([
    F.sum(F.col(c).isNotNull().cast("int")).alias(c)
    for c in mdb.columns
])

# Convert result to dictionary
non_null_counts = null_check.collect()[0].asDict()

# Create table for columns with fill rate = 0
all_null_data = [
    (col, 0.0)
    for col, count in non_null_counts.items()
    if count == 0
]

# Create DataFrame
all_null_df = spark.createDataFrame(
    all_null_data,
    ["column_name", "fill_rate"]
)

# Display table
all_null_df.display()

In [0]:
from pyspark.sql import functions as F

# Total row count
total_rows = mdb.count()

# Count non-null values
null_check = mdb.select([
    F.sum(F.col(c).isNotNull().cast("int")).alias(c)
    for c in mdb.columns
])

non_null_counts = null_check.collect()[0].asDict()

# Columns with fill rate = 0
all_null_cols = [
    col for col, count in non_null_counts.items()
    if count == 0
]

# Drop columns from main DataFrame
mdb_cleaned = mdb.drop(*all_null_cols)

# Verify
print("Dropped columns:")
print(all_null_cols)

mdb_cleaned.printSchema()

In [0]:
(rows, cols) = (mdb.count(), len(mdb.columns))

print("Rows_og:", rows)
print("Columns_og:", cols)

(rows_cleaned,cols_cleaned)= (mdb_cleaned.count(), len(mdb_cleaned.columns))

print("Rows_cleaned:", rows_cleaned)
print("Columns_cleaned:", cols_cleaned)

In [0]:
from pyspark.sql.functions import countDistinct

mdb_cleaned.select(countDistinct("PROVIDER_ID")).show()

In [0]:
print((mdb_cleaned.dropDuplicates(["PROVIDER_ID", "INSTANCE"]).count(),
       len(mdb_cleaned.dropDuplicates(["PROVIDER_ID", "INSTANCE"]).columns)))

In [0]:
mdb_cleaned = mdb_cleaned.drop("ZIP_CODE","ZIPCODE","ORG_NM","EDUCATION_LEVEL","BH_ID","ACCT_NAME","MAJOR_CLASS_OF_TRADE","DESCRIPTION", "MONTH_ID","A_HIGH_POTENTIAL_WRITER_SEGMENT","C_LOWER_POTENTIAL_WRITER_SEGMENT","B_HIGH_POTENTIAL_NON_WRITER_SEGMENT")

In [0]:
(rows, cols) = (mdb_cleaned.count(), len(mdb_cleaned.columns))

print("Rows:", rows)
print("Columns:", cols)

In [0]:
from pyspark.sql.functions import col, concat

mdb_cleaned = (
    mdb_cleaned
    .withColumn("PROVIDER_ID", concat(col("PROVIDER_ID").cast("string"),
                                      col("INSTANCE").cast("string")).cast("int"))
    .drop("INSTANCE")
)

In [0]:
mdb_cleaned.limit(2).display()

In [0]:
mdb_cleaned.select(countDistinct("PROVIDER_ID")).show()

In [0]:
# Create dictionary of datatype -> columns
dtype_dict = defaultdict(list)
 
for col, dtype in mdb_cleaned.dtypes:
    dtype_dict[dtype].append(col)
 
# Convert to list of Rows
data = [
    Row(datatype=dtype, columns=cols)
    for dtype, cols in dtype_dict.items()
]
 
# Create PySpark DataFrame
dtype_df = spark.createDataFrame(data)
 
# Display result
dtype_df.display()

In [0]:
print(mdb_cleaned.schema["TOTAL_NBRX_C4"].dataType)

In [0]:
from pyspark.sql.functions import when, col

# collect distinct gender values
gender_values = [r["GENDER"] for r in mdb_cleaned.select("GENDER").distinct().collect() if r["GENDER"] is not None]

# create one-hot columns
for val in gender_values:
    mdb_cleaned = mdb_cleaned.withColumn(
        f"GENDER_{val}",
        when(col("GENDER") == val, 1.0).otherwise(0.0)
    )

# drop original column
mdb_cleaned = mdb_cleaned.drop("GENDER")

In [0]:
(rows, cols) = (mdb_cleaned.count(), len(mdb_cleaned.columns))
 
print("Rows_og:", rows)
print("Columns_og:", cols)

In [0]:
# Create dictionary of datatype -> columns
dtype_dict = defaultdict(list)
 
for col, dtype in mdb_cleaned.dtypes:
    dtype_dict[dtype].append(col)
 
# Convert to list of Rows
data = [
    Row(datatype=dtype, columns=cols)
    for dtype, cols in dtype_dict.items()
]
 
# Create PySpark DataFrame
dtype_df = spark.createDataFrame(data)
 
# Display result
dtype_df.display()

In [0]:
[c for c in mdb_cleaned.columns if c.startswith("GENDER")]

In [0]:
mdb_cleaned.limit(2).display()

In [0]:
# Create dictionary of datatype -> columns
dtype_dict = defaultdict(list)
 
for col, dtype in mdb_cleaned.dtypes:
    dtype_dict[dtype].append(col)
 
# Convert to list of Rows
data = [
    Row(datatype=dtype, columns=cols)
    for dtype, cols in dtype_dict.items()
]
 
# Create PySpark DataFrame
dtype_df = spark.createDataFrame(data)
 
# Display result
dtype_df.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
 
TARGET_PRECISION = 38
TARGET_SCALE     = 6

target_type      = DecimalType(TARGET_PRECISION, TARGET_SCALE)
 
NUMERIC_PREFIXES = ("int", "bigint", "smallint", "tinyint", "float", "double", "decimal", "byte", "short", "long")
 
cols_to_cast = [
    c for c, dtype in mdb_cleaned.dtypes
    if dtype.lower().startswith(NUMERIC_PREFIXES)

]
 
df_cast = mdb_cleaned.select(*[
    F.col("`" + c + "`").cast(target_type).alias(c) if c in cols_to_cast else F.col("`" + c + "`")
    for c in mdb_cleaned.columns
])
df_cast.printSchema()

In [0]:
# Create dictionary of datatype -> columns
dtype_dict = defaultdict(list)
 
for col, dtype in df_cast.dtypes:
    dtype_dict[dtype].append(col)
 
# Convert to list of Rows
data = [
    Row(datatype=dtype, columns=cols)
    for dtype, cols in dtype_dict.items()
]
 
# Create PySpark DataFrame
dtype_df = spark.createDataFrame(data)
 
# Display result
dtype_df.display()

In [0]:
(rows, cols) = (df_cast.count(), len(df_cast.columns))

print("Rows_og:", rows)
print("Columns_og:", cols)

In [0]:
df_cast.limit(2).display()

In [0]:
# df_cast.write \
#    .format("delta") \
#    .mode("overwrite") \
#    .option("overwriteSchema", "true") \
#    .saveAsTable(
#        "`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.feature_selection_mdb_cleaned_wo_segment_25k")


In [0]:
df_cast = df_cast.drop("A_HIGH_POTENTIAL_WRITER_SEGMENT","C_LOWER_POTENTIAL_WRITER_SEGMENT","B_HIGH_POTENTIAL_NON_WRITER_SEGMENT")

In [0]:
(rows, cols) = (df_cast.count(), len(df_cast.columns))

print("Rows_og:", rows)
print("Columns_og:", cols)

In [0]:
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Driver RAM: {ram_gb:.1f} GB")

In [0]:
# Estimated in-memory size
rows = df_cast.count()
cols = len(df_cast.columns)
estimated_gb = (rows * cols * 8) / (1024**3)  # assuming 8 bytes per value
print(f"Rows: {rows}, Cols: {cols}")
print(f"Estimated size: {estimated_gb:.1f} GB")

In [0]:
# ============================================================
# Optimized Parallel Parquet Write
# ============================================================

import math

n_partitions = 200 
PARQUET_PATH = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

(df_cast
    .repartition(n_partitions)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("parquet.enable.dictionary", "false")  # 27k numeric cols — skip dict encoding
    .option("compression", "snappy")               # fastest compression
    .option("parquet.block.size", 134217728)        # 128MB row groups
    .save(PARQUET_PATH))

print("Parquet saved at:")
print(PARQUET_PATH)



In [0]:
# ============================================================
# Save df_cast as a SINGLE Parquet file
# ============================================================

PARQUET_PATH = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

(df_cast
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("parquet.enable.dictionary", "false")
    .option("compression", "snappy")
    .save(PARQUET_PATH))

print("Parquet saved successfully at:")
print(PARQUET_PATH)

In [0]:
# ============================================================
# Save df_cast as Parquet
# ============================================================

PARQUET_PATH = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_parquet"
)

df_cast.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(PARQUET_PATH)

print("Parquet saved successfully at:")
print(PARQUET_PATH)

**Feature Selection Code**